# Export the papaya cascade for Android

This notebook converts the same YOLOv11m and EfficientNet-B2 checkpoints used by the evaluation notebook. YOLO is exported to NCNN at 832 px, while EfficientNet-B2 is exported to float16 TFLite with float32 RGB input in the 0-255 range.

In [ ]:
# Match the versions recorded by the training notebook and .keras metadata.
%pip install -q -U "ultralytics>=8.3" "tensorflow==2.19.0" "keras==3.13.2"


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import json
import shutil
from pathlib import Path

import numpy as np
import tensorflow as tf
from PIL import Image
from ultralytics import YOLO

PROJECT_ROOT = Path('/content/drive/MyDrive/Papaya_Leaf_Disease_Classification')
EXPORT_ROOT = PROJECT_ROOT / 'Model' / 'Android_Export'
EXPORT_ROOT.mkdir(parents=True, exist_ok=True)
print('TensorFlow:', tf.__version__)
print('Export directory:', EXPORT_ROOT)


In [ ]:
def find_checkpoint(candidates, suffix, required_terms):
    for candidate in candidates:
        if candidate.exists():
            return candidate
    matches = []
    for path in PROJECT_ROOT.rglob(suffix):
        normalized = str(path).lower().replace('_', '').replace('-', '')
        if all(term in normalized for term in required_terms):
            matches.append(path)
    if not matches:
        raise FileNotFoundError(f'Cannot find {suffix} with terms {required_terms}')
    return sorted(matches, key=lambda path: len(str(path)))[0]

YOLO_CKPT = find_checkpoint(
    [
        PROJECT_ROOT / 'Model' / 'YOLOv11' / 'YOLOv11m' / 'weights' / 'best.pt',
        PROJECT_ROOT / 'Model' / 'YOLOv11' / 'yolov11_m' / 'weights' / 'best.pt',
        PROJECT_ROOT / 'outputs' / 'yolo' / 'yolov11_m' / 'weights' / 'best.pt',
    ],
    'best.pt',
    ['yolo', '11m'],
)
CNN_CKPT = find_checkpoint(
    [
        PROJECT_ROOT / 'Model' / 'CNN' / 'EfficientNetB2' / 'best_model.keras',
        PROJECT_ROOT / 'outputs' / 'cnn' / 'efficientnet_b2' / 'best_model.keras',
    ],
    'best_model.keras',
    ['efficientnetb2'],
)

assert YOLO_CKPT.exists(), YOLO_CKPT
assert CNN_CKPT.exists(), CNN_CKPT
print('YOLO checkpoint:', YOLO_CKPT)
print('CNN checkpoint:', CNN_CKPT)


In [ ]:
# Export raw YOLO predictions. NMS and confidence retries run in the Android app.
yolo = YOLO(str(YOLO_CKPT))
exported_path = Path(yolo.export(
    format='ncnn',
    imgsz=832,
    batch=1,
))
if not exported_path.is_absolute():
    exported_path = (YOLO_CKPT.parent / exported_path).resolve()

param_files = list(exported_path.rglob('*.param'))
bin_files = list(exported_path.rglob('*.bin'))
if len(param_files) != 1 or len(bin_files) != 1:
    raise RuntimeError(f'Unexpected NCNN export contents: {exported_path}')

YOLO_PARAM_OUT = EXPORT_ROOT / 'yolo11m_papaya.param'
YOLO_BIN_OUT = EXPORT_ROOT / 'yolo11m_papaya.bin'
shutil.copy2(param_files[0], YOLO_PARAM_OUT)
shutil.copy2(bin_files[0], YOLO_BIN_OUT)
print('NCNN files:', YOLO_PARAM_OUT, YOLO_BIN_OUT)


In [ ]:
cnn = tf.keras.models.load_model(CNN_CKPT, compile=False)
print('Keras input:', cnn.input_shape, 'output:', cnn.output_shape)
if tuple(cnn.input_shape[1:]) != (260, 260, 3) or cnn.output_shape[-1] != 5:
    raise ValueError('Expected EfficientNet-B2 input [1,260,260,3] and output [1,5]')

converter = tf.lite.TFLiteConverter.from_keras_model(cnn)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.float16]
tflite_bytes = converter.convert()

TFLITE_OUT = EXPORT_ROOT / 'efficientnet_b2_papaya_float16.tflite'
TFLITE_OUT.write_bytes(tflite_bytes)
print('TFLite model:', TFLITE_OUT, f'({TFLITE_OUT.stat().st_size / 1e6:.1f} MB)')


In [ ]:
# Validate tensor shapes and compare one image when test data is available.
interpreter = tf.lite.Interpreter(model_path=str(TFLITE_OUT), num_threads=4)
interpreter.allocate_tensors()
input_info = interpreter.get_input_details()[0]
output_info = interpreter.get_output_details()[0]
print('TFLite input:', input_info['shape'], input_info['dtype'])
print('TFLite output:', output_info['shape'], output_info['dtype'])
assert list(input_info['shape']) == [1, 260, 260, 3]
assert list(output_info['shape']) == [1, 5]
assert input_info['dtype'] == np.float32

test_roots = [
    PROJECT_ROOT / 'prepared_data_v1' / 'cnn_dataset' / 'test',
    PROJECT_ROOT / 'Data' / 'prepared_data_v1' / 'cnn_dataset' / 'test',
]
test_images = []
for root in test_roots:
    if root.exists():
        test_images = [p for p in root.rglob('*') if p.suffix.lower() in {'.jpg', '.jpeg', '.png'}]
        if test_images:
            break

if test_images:
    sample = np.asarray(Image.open(test_images[0]).convert('RGB').resize((260, 260)), dtype=np.float32)[None, ...]
    keras_probs = cnn.predict(sample, verbose=0)[0]
    interpreter.set_tensor(input_info['index'], sample)
    interpreter.invoke()
    tflite_probs = interpreter.get_tensor(output_info['index'])[0]
    print('Validation image:', test_images[0])
    print('Keras argmax/conf:', int(np.argmax(keras_probs)), float(np.max(keras_probs)))
    print('TFLite argmax/conf:', int(np.argmax(tflite_probs)), float(np.max(tflite_probs)))
    print('Maximum probability difference:', float(np.max(np.abs(keras_probs - tflite_probs))))
else:
    print('No test image found; tensor shape validation completed.')


In [ ]:
# Canonical names exactly match notebook 06; Android adds spaces only for display.
cnn_labels = ['Anthracnose', 'BacterialSpot', 'Curl', 'Healthy', 'RingSpot']
yolo_labels = ['Anthracnose', 'BacterialSpot', 'Curl', 'RingSpot']
(EXPORT_ROOT / 'papaya_cnn_labels.txt').write_text('\n'.join(cnn_labels) + '\n', encoding='utf-8')
(EXPORT_ROOT / 'papaya_yolo_labels.txt').write_text('\n'.join(yolo_labels) + '\n', encoding='utf-8')

config = {
    'yolo_input_size': 832,
    'cnn_input_size': 260,
    'yolo_classes': yolo_labels,
    'cnn_classes': cnn_labels,
    'initial_yolo_confidence': 0.50,
    'retry_yolo_confidences': [0.45, 0.40, 0.35],
    'yolo_nms_iou': 0.70,
    'full_image_cnn_disease_min_probability': 0.50,
    'bbox_expand_ratio_each_side': 0.10,
    'image_vote': 'sum(cnn_confidence * yolo_confidence) by predicted CNN class',
}
CONFIG_OUT = EXPORT_ROOT / 'papaya_android_config.json'
CONFIG_OUT.write_text(json.dumps(config, indent=2), encoding='utf-8')

archive_base = EXPORT_ROOT / 'papaya_android_models'
old_archive = archive_base.with_suffix('.zip')
if old_archive.exists():
    old_archive.unlink()
archive_path = Path(shutil.make_archive(str(archive_base), 'zip', EXPORT_ROOT))
print('Package:', archive_path)
print('\nCopy these three files to AndroidApp/app/src/main/assets:')
print(YOLO_PARAM_OUT.name)
print(YOLO_BIN_OUT.name)
print(TFLITE_OUT.name)


In [ ]:
# Optional browser download. The files remain in Google Drive even if this is skipped.
from google.colab import files
files.download(str(EXPORT_ROOT / 'papaya_android_models.zip'))
